In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), override=True)

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("Add GROQ_API_KEY to the .env file before running this notebook.")

GROQ_MODEL = os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b")
MAX_OUTPUT_TOKENS = int(os.getenv("GROQ_MAX_TOKENS", "256"))

## Completion model

In [2]:
from langchain_groq import ChatGroq

llmModel = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.2,
    reasoning_effort="none",
    max_tokens=MAX_OUTPUT_TOKENS,
)

# chat completion model

In [3]:
chatModel = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.2,
    reasoning_effort="none",
    max_tokens=MAX_OUTPUT_TOKENS,
)

## Prompts and Prompt Templates

In [4]:
# PromptTemplate with the Groq chat model
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} story about {topic}"
)

llmModelPrompt = prompt_template.format(
    adjective="curious",
    topic="Tesla"
)

res = llmModel.invoke(llmModelPrompt)
print(res.content)

Here is a curious, often overlooked story about Nikola Tesla that blends his genius, his eccentricities, and a touch of mystery.

### The Pigeon and the "Love" That Wasn’t Human

In the final years of his life, living alone in the New Yorker Hotel in Manhattan, Nikola Tesla became increasingly reclusive and eccentric. While the world knew him as the inventor of alternating current (AC) and the man who lit up the world, few knew about his most intimate companion: **a white pigeon**.

Tesla had a deep, almost spiritual connection with this particular bird. He would feed it, talk to it, and even kiss it on the head. When asked about this unusual attachment, Tesla reportedly said:

> *"I love her. She is a beautiful bird. She is the only one who ever loved me."*

But here’s the curious twist: **Tesla didn’t just love *one* pigeon.**

#### The Pattern
Historians and biographers have noted that Tesla had a recurring pattern of forming intense, one-sided emotional bonds with specific pigeons 

In [5]:
# chat completion model: mostely use chat completion models.
# They are designed to handle multi-turn conversations effectively.

from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an {profession} expert on {topic}."),
        ("human", "Hello, Mr. {profession}, can you please answer a question?"),
        ("ai", "Sure!"),
        ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(
    profession="Historian",
    topic="The Kennedy family",
    user_input="How many grandchildren had Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)
print(response.content)


Joseph P. Kennedy had **12 grandchildren**.

Here is the breakdown by child:

1.  **Rosemary Kennedy** (1918–2005): Had **1** child (Michael J. Kennedy).
2.  **Eunice Kennedy Shriver** (1921–1995): Had **4** children (Mark, Timothy, Toni, and Maria Shriver).
3.  **John F. Kennedy** (1917–1963): Had **3** children (Caroline, John Jr., and Patrick Bouvier Kennedy). *Note: Patrick Bouvier died in infancy, but he is counted as a child/grandchild.*
4.  **Robert F. Kennedy** (1925–1968): Had **6** children (Michael, Douglas, Robert Jr., Jennifer, John, and Kathleen Kennedy).
5.  **Eugene J. Kennedy** (1927–2009): Had **1** child (Eugene J. Kennedy Jr.).
6.  **Jean Kennedy Smith** (1934–2023): Had **2** children (John and


# Old way

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content=(
                "You are an Historian expert on the Kennedy family."
            )
        ),
        HumanMessagePromptTemplate.from_template("{user_input}"),
    ]
)

messages = chat_template.format_messages(
    user_input="Name the children and grandchildren of Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)

In [ ]:
response

In [ ]:
print(response.content)

# Few Shot Prompting
### Few-shot prompting is a technique where an AI model is given a small number of input-output examples to demonstrate the expected pattern before processing a new input.
Translate English to Spanish:

English: Hello
Spanish: Hola

English: Goodbye
Spanish: Adiós

English: Thank you
Spanish:

In [6]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# Chains

# Chains in LangChain

A chain connects multiple processing steps.

The output of one step becomes the input of the next step.

## Basic flow

User input → Prompt → Model → Output parser → Final result

## Example

```python
chain = prompt | model | parser

result = chain.invoke({
    "question": "What is Python?"
})

In [7]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)


chain = final_prompt | chatModel

res = chain.invoke({"input": "How are you?"})
print(res.content)

¿Cómo estás?


In [ ]:
res

# LangChain Expression Language (LCEL)

LangChain Expression Language, or LCEL, is a simple way to connect LangChain components into a chain.

It uses the pipe operator (`|`) to pass the output of one component to the next component.

## Basic flow

User input → Prompt → Model → Output parser → Final result

## Basic syntax

```python
chain = prompt | model | parser

In [ ]:
# Langchain Expressions Language

# Output Parsers
An output parser converts the model's response into the required format.

## Basic flow

Prompt → Model → Output parser → Final result

### 1. String output

`StrOutputParser` converts the model response into plain text.

```python
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = prompt | model | parser
```

### 2. JSON output

`JsonOutputParser` converts the model response into a Python dictionary.

```python
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

chain = prompt | model | parser
```

## Simple definition

An output parser converts raw model output into a useful format, such as a string, dictionary, or Pydantic object.

In [9]:
llmModel = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.2,
    reasoning_effort="none",
    max_tokens=MAX_OUTPUT_TOKENS,
)

In [10]:
chatModel = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.2,
    reasoning_effort="none",
    max_tokens=MAX_OUTPUT_TOKENS,
)

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import SimpleJsonOutputParser

json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an `answer` key that answers the following question: {question}"
)

json_parser = SimpleJsonOutputParser()

json_chain = json_prompt | llmModel | json_parser

In [12]:
json_parser.get_format_instructions()

'Return a JSON object.'

In [13]:
res  = json_chain.invoke({"question": "What is the biggest country?"})

In [14]:
res

{'answer': 'Russia'}

In [15]:
type(res)

dict

#### Optionally, you can use Pydantic to define a custom output format

In [16]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

In [17]:
# Define a Pydantic Object with the desired output format.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [18]:
# Define the parser referring the Pydantic Object
parser = JsonOutputParser(pydantic_object=Joke)

# Add the parser format instructions in the prompt definition.
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Create a chain with the prompt and the parser
chain = prompt | chatModel | parser

chain.invoke({"query": "Tell me a joke."})

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}